## Practice Assignment - WildFire Activities in Australia 2

In [1]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from dash import no_update
import datetime as dt


In [2]:
# Read the wildfire data into pandas dataframe
df = pd.read_csv("data/Historical_Wildfires.csv",date_format='%m/%d/%Y', parse_dates=["Date"])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df.dtypes

Region                                            str
Date                                   datetime64[us]
Estimated_fire_area                           float64
Mean_estimated_fire_brightness                float64
Mean_estimated_fire_radiative_power           float64
Mean_confidence                               float64
Std_confidence                                float64
Var_confidence                                float64
Count                                           int64
Replaced                                          str
Year                                            int32
Month                                           int32
dtype: object

In [3]:
#Create app
app = dash.Dash(__name__)

# Clear the layout and do not display exception till callback gets executed
app.config.suppress_callback_exceptions = True

#Layout Section of Dash
#Task 2.1 Add the Title to the Dashboard
app.layout = html.Div(children=[html.H1("Australia Wildfire Dashboard", style={'textAlign':'center', 'color':'#5030D36', 'font-size':35}),
        html.Div([
            html.Div([
                html.H2('Select Region:', style={'margin-right':'2em'}),
                dcc.RadioItems(
                    [
                        {"label":"New South Wales","value": "NSW"},
                        {"label":"Northern Territory","value":"NT"},
                        {"label":"Queensland","value":"QL"},
                        {"label":"South Australia","value":"SA"},
                        {"label":"Tasmania","value":"TA"},
                        {"label":"Victoria","value":"VI"},
                        {"label":"Western Australia","value":"WA"},
                    ], value="NSW", id='region', inline=True
                )
            ]),
            html.Div([
                html.H2('Select Year:', style={'margin-right':'2em'}),
                dcc.Dropdown(df.Year.unique(), value=2005, id='year', style={'width':'10%','height':'50px', 'font-size':26})]
            ),
            html.Div([
                html.Div([ ], id='plot1'),
                html.Div([ ], id='plot2')
            ], style={'display':'flex'} ),
        ])
    ],
        style={
            'backgroundColor':"#FFFFFF", 
            'minHeight':'100vh', # Ensures the background covers the full screen height
            'padding':'20px'
        }
    )

#TASK 2.4: Add the Ouput and input components inside the app.callback decorator.
@app.callback(
    [Output(component_id='plot1', component_property='children'),
     Output(component_id='plot2', component_property='children')],
    [Input(component_id='region', component_property='value'),
     Input(component_id='year', component_property='value')]
)

#TASK 2.5: Add the callback function.
def reg_year_display(input_region,input_year):
    region_data = df[df['Region'] == input_region]
    y_r_data = region_data[region_data['Year']==input_year]
   
    #Plot one - Monthly Average Estimated Fire Area 
    est_data = y_r_data.groupby("Month")['Estimated_fire_area'].mean().reset_index()
    fig1 = px.pie(est_data, values='Estimated_fire_area', names='Month', title="{} : Monthly Average Estimated Fire Area in year {}".format(input_region,input_year))
   
    #Plot two - Monthly Average Count of Pixels for Presumed Vegetation Fires
    veg_data = y_r_data.groupby("Month")['Count'].mean().reset_index()
    fig2 = px.bar(veg_data, x='Month', y='Count', title='{} : Average Count of Pixels for Presumed Vegetation Fires in year {}'.format(input_region,input_year))
    
    return [dcc.Graph(figure=fig1), dcc.Graph(figure=fig2)]



if __name__ == '__main__':
    app.run(jupyter_mode="inline") # "inline" "external" "jupyterlab"